# La figura UMAP del espacio CLIP de imagen

**Extra comprometido la tabla de tokens, costo y la figura · objetivo el objetivo opcional de la figura UMAP · pendiente `el registro de decisiones el criterio de idioma`.**

Proyecta a dos dimensiones los **2.605 vectores CLIP de imagen de las obras indexadas** y las
colorea por `culture`, `year`, **tema** y `style`, para responder la pregunta de `la teoría aplicada § UMAP`:

> *"Si los clusters se forman por **tema** (paisajes juntos, retratos juntos) y no por
> **estilo**, esa figura es la evidencia visual del desafío central del TP."*

---

### Qué hace este notebook y qué **no** hace

| | |
|---|---|
| ✅ **Lee** | el `ManifiestoDeSplit` (quién está indexado), el `CatálogoCanónico` (la metadata) y el **caché de embeddings de imagen** (los vectores) |
| ✅ **Deriva** | una etiqueta de **tema** a partir del **título escrito por el museo**, y un número de separación por etiquetado |
| ❌ **No recalcula** | ningún embedding. No llama a ninguna API. No necesita la key |
| ❌ **No toca Chroma** | la variante **no está registrada** (espera la key) y el caché tiene los vectores con su identidad completa |
| ❌ **No cierra el criterio de normalización de culture** | el agrupamiento de `culture` de esta figura es **un dato declarado de la figura**, no una decisión del proyecto |
| ❌ **No etiqueta el tema con CLIP** | ver los tres ciclos de vida. Sería circular y es la razón por la que este notebook existe |

> ⚠️ **Desvío declarado respecto del módulo de evaluación.** Esa sección dice que la figura es
> *"sobre el conjunto indexado de **una variante**"* y que el criterio de `culture` se lee **de la
> receta de la variante**. **No hay variante registrada**, así que acá la membresía se lee del
> **`ManifiestoDeSplit`** —que es su dueño (la regla de qué se versiona), no la variante— y el criterio de `culture`
> de `config/default.toml`. La proyección en sí no depende de la variante: los vectores están
> indexados por `(obra, hash de bytes, checkpoint, versión del preprocesado)`. **Lo que habría
> que rehacer cuando la variante exista es sólo el panel de `culture`**, y sólo si la receta
> declara otro criterio.

## 0 · Parámetros y procedencia

Todo lo que hace falta para re-correr esta figura y obtener **la misma imagen**.
los criterios de entrega el dataset documentado y reproducible pide que lo publicado sea reproducible; sin la semilla fijada, UMAP
cambia en cada corrida y la figura deja de ser citable.

In [1]:
from pathlib import Path
import json, re, collections, platform, sys

import numpy as np

# ── Procedencia declarada ───────────────────────────────────────────────────
SPLIT_ID    = "5b6cba32126fed61"          # artifacts/splits/<split_id>.json
CATALOGO_ID = "e10c4e4f9cce0e74"          # data/catalogo/<catalogo_id>/
CHECKPOINT  = "openai/clip-vit-base-patch32"   # el stack declarado · el registro de decisiones D4
PREPROC     = "1"                          # version_del_preprocesado

# `config/default.toml § [split]`: la semilla del proyecto es el número de la
# materia (73.69) sin el punto. **No es una fecha a propósito.**
SEMILLA = 7369

# ── Parámetros de UMAP, declarados ──────────────────────────────────────────
#  `metric="cosine"` es explícito y no el default: los vectores están
#  normalizados a norma 1 (`el registro de decisiones la verificación de la norma de los vectores`), así que coseno, producto interno y
#  euclídea inducen el mismo orden — pero el documento que acompaña la figura
#  tiene que decir cuál se usó, no dejarlo al default de la librería.
UMAP_PARAMS = dict(
    n_components=2,
    n_neighbors=15,      # default de umap-learn, declarado
    min_dist=0.1,        # default de umap-learn, declarado
    metric="cosine",     # explícito, no default (el default es "euclidean")
    random_state=SEMILLA,
)

# k de la métrica de separación de la las decisiones señalizadas
K_VECINOS = 10

RAIZ = Path.cwd()
while not (RAIZ / "pyproject.toml").exists():
    RAIZ = RAIZ.parent
SALIDA = RAIZ / "artifacts" / "informes"
print("raíz del repo:", RAIZ)
print("salida:", SALIDA)
print("python:", platform.python_version())

raíz del repo: /home/matias/Documents/ITBA/LLM/LLM-TP3
salida: /home/matias/Documents/ITBA/LLM/LLM-TP3/artifacts/informes
python: 3.10.12


## 1 · De dónde salen los datos

🔴 **La figura es sobre las 2.605 indexadas, no sobre las 5.751 embebidas.** Es explícito en
tres documentos: `la teoría aplicada § UMAP` (*"**no** el universo elegible … **ni** el elegible menos el
held-out"*), `el registro de decisiones el criterio de idioma` (*"sobre las obras **indexadas**"*) y la tabla de tokens, costo y la figura
(*"una figura sobre el **conjunto indexado**"*). La tentación va en la otra dirección —las
5.751 están todas embebidas y salen de un `glob`— y **sería otra figura y diría otra cosa**.

| Qué | De dónde |
|---|---|
| **Quiénes** (2.605 ids) | el lado corpus de `artifacts/splits/5b6cba32126fed61.json` |
| **La metadata** | `data/catalogo/e10c4e4f9cce0e74/registros.jsonl` |
| **Los vectores** | `data/embeddings-imagen/<checkpoint>/<preproc>/<artwork_id>-<hash>.npy` |

Se usan los lectores del propio paquete (`tp3_rag`), que **re-verifican la identidad al leer**:
el `CatálogoCanónico` rehace el hash de los 5.915 registros contra el declarado en
`catalogo.json`, y el `ManifiestoDeSplit` valida que sus tres conteos cierren contra sus
propios ids.

In [2]:
sys.path.insert(0, str(RAIZ / "src"))
from tp3_rag.dominio.catalogo import CatálogoCanónico
from tp3_rag.dominio.split import ManifiestoDeSplit
from tp3_rag.puertos.almacenamiento import AlmacenamientoEnArchivos
from tp3_rag.modulo_a.normalizador_de_culture import NormalizadorDeCulture
from tp3_rag.nucleo.configuracion import Configuración

almacén = AlmacenamientoEnArchivos(RAIZ)

manifiesto = ManifiestoDeSplit.leer(almacén, "artifacts/splits", SPLIT_ID)
catálogo   = CatálogoCanónico.cargar(almacén, Path("data/catalogo") / CATALOGO_ID)

assert manifiesto.catalogo_id == catálogo.catalogo_id == CATALOGO_ID, "el split no salió de este catálogo"

ids = list(manifiesto.ids_corpus)
por_id = {r.artwork_id: r for r in catálogo.registros}
regs = [por_id[a] for a in ids]

print("catálogo verificado :", catálogo.catalogo_id, "·", len(catálogo.registros), "registros del universo")
print("split               :", manifiesto.split_id, "·", manifiesto.conteos)
print("obras de la figura  :", len(regs))
assert len(regs) == manifiesto.conteos["corpus_indexado"] == 2605

catálogo verificado : e10c4e4f9cce0e74 · 5915 registros del universo
split               : 5b6cba32126fed61 · {'corpus_indexado': 2605, 'elegibles_con_texto_e_imagen': 2894, 'held_out': 289}
obras de la figura  : 2605


In [3]:
def _seguro(v: str) -> str:
    """El mismo saneo que usa `CachéDeEmbeddingsDeImagen.ruta` (`embebedor_de_imagenes.py`)."""
    return v.replace("/", "__").replace(":", "_")

raíz_caché = RAIZ / "data" / "embeddings-imagen" / _seguro(CHECKPOINT) / PREPROC
V = np.stack([
    np.load(raíz_caché / f"{_seguro(r.artwork_id)}-{_seguro(r.imagen.hash_bytes)}.npy")
    for r in regs
]).astype(np.float64)

# `el registro de decisiones la verificación de la norma de los vectores` · verificado acá en vez de supuesto
normas = np.linalg.norm(V, axis=1)
print("matriz             :", V.shape, "(obras × dims)")
print("dim                :", V.shape[1], "— ViT-B/32")
print("‖v‖ min/máx: %.6f / %.6f (la verificación de la norma de los vectores: norma 1)" % (normas.min(), normas.max()))
assert V.shape == (2605, 512)
assert np.allclose(normas, 1.0, atol=1e-5)

matriz             : (2605, 512) (obras × dims)
dim                : 512 — ViT-B/32
‖v‖ min/máx        : 1.000000 / 1.000000  (D8: norma 1)


## 2 · 🔴 La trampa: cómo se etiqueta el **tema** sin usar CLIP

**"Colorear por tema" es el problema difícil, porque `tema` no es un campo.** `culture` y `year`
están en la metadata; *paisaje, retrato, bodegón* **no existen en el esquema canónico** — es el
mismo hueco que el diseño atributos y el que `el registro de decisiones el criterio de idioma` deja abierto.

> ⚠️ **La salida tentadora es la peor: etiquetar el tema con CLIP zero-shot.** Es la técnica de
> la Clase 9 y funcionaría —embeber *"a painting of a landscape"*, *"a portrait"*, y asignar
> cada obra al prompt más cercano— **pero es circular**: estaríamos etiquetando los datos **con
> el mismo espacio que la figura pretende evaluar**. Los clusters coincidirían con las etiquetas
> **por construcción**, y la figura no probaría nada. Sería la peor clase de evidencia: la que
> parece fuerte.

**La vía honesta: derivar el tema del `title`, que lo escribió el museo.** Es un campo de
catálogo, redactado antes de que CLIP existiera y sin ninguna relación con los píxeles salvo la
que pasa por el ojo de quien catalogó. *"Landscape…"*, *"Portrait of…"*, *"Still Life with…"*,
*"View of…"*.

### Las reglas del léxico, declaradas

1. Se busca con **expresiones regulares sobre el título en minúsculas**, con frontera de palabra.
2. **El orden de la tabla es la precedencia**: gana el primer tema que machea. *"Landscape with
   the Penitent Saint Jerome"* queda en `paisaje` y no en `escena religiosa` porque `paisaje`
   viene antes — y esa decisión está declarada acá, no escondida.
3. Una obra que no machea ningún patrón **queda sin etiquetar y se dibuja en gris**, contada.
4. `escena religiosa` **une lo cristiano y lo devocional asiático a propósito**. Separarlos
   habría convertido un eje de *tema* en un eje de *cultura*, que es exactamente la confusión
   que esta figura tiene que poder distinguir. Que una clase culturalmente heterogénea igual
   se mantenga junta es un resultado **más** fuerte, no más débil.

**Es cruda y cubre una fracción de las obras, y eso está bien:** una etiqueta independiente
sobre un tercio del corpus prueba más que una etiqueta circular sobre el 100 %.

In [4]:
# ── El léxico. El orden ES la precedencia. ──────────────────────────────────
LÉXICO_TEMA = (
    ("bodegón",          r"\bstill[- ]lifes?\b|\bstill life\b"),
    ("retrato",          r"\bportraits?\b|\bself-?portrait\b"),
    ("paisaje",          r"\blandscapes?\b|\bviews?\s+(of|from|near|at|on|in|over)\b"
                         r"|\bscenery\b|\bpanorama\b|\bseascape\b"),
    ("desnudo",          r"\bnudes?\b|\bbathers?\b"),
    # `\bsaints?\b(?!-)`: el guión descarta los topónimos franceses —`Gare
    # Saint-Lazare`, `Saint-Rémy`, `Saint-Jean-du-Doigt`—, que eran 10 falsos
    # positivos. Es una **regla declarada**, no una lista de excepciones curada
    # obra por obra: si fuera una lista, dejaría de ser auditable.
    ("escena religiosa", r"\bchrist\b|\bvirgin\b|\bmadonna\b|\bcrucifix\w*\b|\bsaints?\b(?!-)|\bst\.\s"
                         r"|\bannunciation\b|\bnativity\b|\bholy\b|\bapostles?\b|\bbaptist\b"
                         r"|\bpiet[aà]\b|\baltarpieces?\b|\baltars?\b"
                         r"|\bbuddh\w*\b|\bbodhisattva\b|\bkannon\b|\bkrishna\b|\bvishnu\b"
                         r"|\bshiva\b|\bdaoist\b|\bjain\b|\bdeit\w+\b|\bguanyin\b"
                         r"|\bavalokit\w*\b|\bmandala\b"),
    ("flora y fauna",    r"\bflowers?\b|\bbamboo\b|\bpeon\w+\b|\blotus\b|\bchrysanthem\w*\b"
                         r"|\borchids?\b|\bblossoms?\b|\bbouquet\b|\bfloral\b"
                         r"|\bbirds?\b|\bhorses?\b|\btigers?\b|\bcats?\b|\bdogs?\b|\bdeer\b"
                         r"|\bmonkeys?\b|\bcranes?\b|\bfish\b|\binsects?\b|\belephants?\b"),
)
_TEMA = [(n, re.compile(p)) for n, p in LÉXICO_TEMA]
TEMAS = [n for n, _ in LÉXICO_TEMA]

def tema_de(título: str | None):
    t = (título or "").lower()
    for nombre, rx in _TEMA:
        if rx.search(t):
            return nombre
    return None

tema = np.array([tema_de(r.title) for r in regs], dtype=object)

n_tot = len(regs)
n_tema = int(sum(t is not None for t in tema))
print("COBERTURA DEL ETIQUETADO DE TEMA")
print("  %-18s %5s %7s" % ("tema", "n", "% corpus"))
for t in TEMAS:
    n = int((tema == t).sum())
    print("  %-18s %5d %6.1f %%" % (t, n, 100 * n / n_tot))
print("  %-18s %5d %6.1f %%" % ("(sin etiquetar)", n_tot - n_tema, 100 * (n_tot - n_tema) / n_tot))
print("  " + "-" * 32)
print("  %-18s %5d %6.1f %%" % ("COBERTURA", n_tema, 100 * n_tema / n_tot))

COBERTURA DEL ETIQUETADO DE TEMA
  tema                   n % corpus
  bodegón               30    1.2 %
  retrato              175    6.7 %
  paisaje              172    6.6 %
  desnudo                8    0.3 %
  escena religiosa     383   14.7 %
  flora y fauna        201    7.7 %
  (sin etiquetar)     1636   62.8 %
  --------------------------------
  COBERTURA            969   37.2 %


### Auditoría de precisión del léxico

Una etiqueta derivada por regex puede equivocarse, y la figura se apoya en ella. Abajo va una
muestra sistemática (cada *n*-ésima obra de cada clase) para que la precisión se pueda **mirar**
en vez de suponer. Los falsos positivos conocidos quedan nombrados en la §8.

In [5]:
for t in TEMAS:
    v = [r for r, et in zip(regs, tema) if et == t]
    paso = max(1, len(v) // 8)
    print(f"── {t}  (n = {len(v)})")
    for r in v[::paso][:8]:
        print(f"     [{r.museum}] {r.title[:96]}")
    print()
sin = [r for r, et in zip(regs, tema) if et is None]
print(f"── sin etiquetar  (n = {len(sin)})")
for r in sin[::max(1, len(sin) // 10)][:10]:
    print(f"     [{r.museum}] {r.title[:96]}")

── bodegón  (n = 30)
     [aic] Still Life—Strawberries, Nuts, &c.
     [aic] Still Life
     [aic] Fish (Still Life)
     [aic] Still Life No. 15
     [aic] Still Life: Corner of a Table
     [cma] Still Life with Rayfish
     [cma] Still Life with Vegetables, Partridge, and a Jug
     [cma] Still Life with Herring, Bread, and Cheese

── retrato  (n = 175)
     [aic] Portrait of a Young Woman
     [aic] Study for "Arrangement in Grey and Black, No. 2: Portrait of Thomas Carlyle"
     [cma] Oval portrait of a woman in a Chaghtai hat
     [cma] Portrait of Juan Maria Osorio
     [cma] Portrait of Infante Don Luis de Borbon
     [cma] Paintings after Ancient Masters: Portrait of Zhongqing in a Landscape
     [cma] Layered Peaks and Splashing Waterfall: Portrait of Hua Yan as a Young Man
     [cma] Posthumous portrait of the Mughal emperor Muhammad Shah (reigned 1719–1748) holding a falcon (re

── paisaje  (n = 172)
     [aic] Landscape with the Penitent Saint Jerome
     [cma] Princes hu

## 3 · Los otros tres coloreados

### `culture` — 102 etiquetas no son coloreables

el criterio de normalización de culture sigue **abierto**. Lo que esta figura hace es **declarar un agrupamiento propio**,
no cerrarlo:

- se normaliza con el criterio de `config/default.toml § [normalizacion.culture]`
  (`primer_componente_por_coma`), que es el mismo punto de variación que implementó;
- se colorean las **top-8 por `n`**, el resto va a **`otras`**, y las obras **sin ningún valor
  de `culture`** van en gris y contadas;
- `culture_raw` es una **lista** (**NR2**): una obra puede traer varias etiquetas. Cuando trae
  más de una del top-8, gana la de mayor `n` — y eso también está declarado.

> 🔴 **Esto no cierra el criterio de normalización de culture.** El agrupamiento de arriba es un corte por frecuencia, no
> una tabla de sinónimos: `China`/`Chinese`, `Netherlands`/`Netherlandish` y la India repartida
> en once etiquetas **siguen sin unirse**, que es exactamente lo que el criterio de normalización tiene abierto
> (`vocabulario_de_culture.GRUPOS_PROPUESTOS`). Un corte por frecuencia es
> lo que la figura necesita para tener colores legibles; **no es una decisión del proyecto**.

### `year` — continuo, escala secuencial

Cubre el **99,97 %**: falta una sola obra (`aic:45753`, con `date_start`/`date_end` en `null`),
que va en gris. Se colorea con una **rampa de un solo tono claro→oscuro**, no con una escala
categórica ni con un arcoíris.

### `style` — el contraste que cierra el argumento

`style` existe sólo en AIC y es **igualmente independiente de CLIP**: lo escribió el museo. Si el
espacio separa temas y **no** separa estilos, la hipótesis queda demostrada con dos coloreados
que usan dos campos de catálogo.

`el relevamiento § El ruido de las etiquetas de estilo` advierte que `style` trae ruido —*"la
superposición entre impresionismo y postimpresionismo"*—, así que no hay que esperar fronteras
limpias. **Lo que se encontró al mirar los valores es peor que eso y se declara acá**: ver §8.

In [6]:
cfg = Configuración.cargar(RAIZ / "config" / "default.toml")
CRITERIO_CULTURE = cfg["normalizacion.culture.criterio"]
normalizador = NormalizadorDeCulture(CRITERIO_CULTURE)

etqs_cult = [normalizador.normalizar(r.culture_raw) for r in regs]
conteo_cult = collections.Counter(e for es in etqs_cult for e in es)
TOP8 = [e for e, _ in conteo_cult.most_common(8)]

def cultura_de(es):
    if not es:
        return None                      # sin ningún valor de `culture` → gris
    for e in TOP8:                       # precedencia declarada: top-8 por `n` descendente
        if e in es:
            return e
    return "otras"

cultura = np.array([cultura_de(es) for es in etqs_cult], dtype=object)
CLASES_CULTURA = TOP8 + ["otras"]

year  = np.array([r.year if r.year is not None else np.nan for r in regs], dtype=float)
style = np.array([r.style or None for r in regs], dtype=object)

n_sin_cult = int(sum(e is None for e in cultura))
print("CULTURE · criterio declarado:", CRITERIO_CULTURE)
print("  etiquetas distintas en el corpus :", len(conteo_cult), " (el criterio de normalización de culture abierto)")
print("  obras sin ningún valor           :", n_sin_cult)
print("  agrupamiento de esta figura      : top-8 + `otras` + gris")
for e in TOP8:
    print("     %-16s %4d" % (e, conteo_cult[e]))
print("     %-16s %4d" % ("otras", int((cultura == "otras").sum())))
print()
print("YEAR   · con valor: %d / %d = %.2f %%  (falta: %s)"
      % (int((~np.isnan(year)).sum()), n_tot, 100 * (~np.isnan(year)).sum() / n_tot,
         ", ".join(r.artwork_id for r, y in zip(regs, year) if np.isnan(y))))
print("         rango: %d – %d" % (np.nanmin(year), np.nanmax(year)))
print()
n_style = int(sum(s is not None for s in style))
print("STYLE  · con valor: %d / %d = %.1f %% del corpus" % (n_style, n_tot, 100 * n_style / n_tot))
for m in ("aic", "cma"):
    sub_m = [s for r, s in zip(regs, style) if r.museum == m]
    print("         %s: %d / %d = %.1f %%"
          % (m, sum(s is not None for s in sub_m), len(sub_m),
             100 * sum(s is not None for s in sub_m) / len(sub_m)))
print("         valores distintos:", len({s for s in style if s}))

CULTURE · criterio declarado: primer_componente_por_coma
  etiquetas distintas en el corpus : 101  (el criterio de normalización de `culture`, abierto)
  obras sin ningún valor           : 11
  agrupamiento de esta figura      : top-8 + `otras` + gris
     France            371
     China             367
     Japan             341
     Mughal India      247
     Italy             193
     Netherlands        99
     Northern India     95
     United States      93
     otras             788

YEAR   · con valor: 2604 / 2605 = 99.96 %  (falta: aic:45753)
         rango: -1295 – 1949

STYLE  · con valor: 567 / 2605 = 21.8 % del corpus
         aic: 567 / 638 = 88.9 %
         cma: 0 / 1967 = 0.0 %
         valores distintos: 39


### Lo que los valores de `style_title` son en realidad

el relevamiento advertía de *fronteras difusas entre movimientos*. Mirando los 39 valores que
efectivamente aparecen, el problema es de otra naturaleza: **la mayoría de los valores no son
estilos**. Se clasifican acá en tres familias, **con la tabla declarada** para que cualquiera
pueda discutirla.

In [7]:
# Clasificación declarada de los valores de `style_title`. Es un juicio nuestro
# sobre un campo del museo, no una normalización del proyecto: va acá para que
# se pueda discutir, y no cierra ningún pendiente.
FAMILIA_STYLE = {
    # — movimiento o escuela artística (lo que "estilo" debería significar) —
    "Impressionism": "movimiento", "Post-Impressionism": "movimiento",
    "Realism": "movimiento", "Renaissance": "movimiento", "Modernism": "movimiento",
    "Folk Art": "movimiento", "Neoclassicism": "movimiento", "Baroque": "movimiento",
    "Mannerism": "movimiento", "Hudson River School": "movimiento",
    "Pre-Raphaelite": "movimiento", "Pointillism": "movimiento", "synthetist": "movimiento",
    # — período, no estilo —
    "17th Century": "período", "18th Century": "período", "16th Century": "período",
    "15th century": "período", "14th century": "período", "13th century": "período",
    "19th century": "período", "20th Century": "período", "nineteenth century": "período",
    "sixteenth century": "período", "new kingdom": "período",
    "third intermediate period": "período",
    # — cultura o lugar, no estilo —
    "Japanese (culture or style)": "cultura", "Chinese (culture or style)": "cultura",
    "Islamic (culture or style)": "cultura", "Indian (South Asian)": "cultura",
    "South Asian": "cultura", "Himalayan": "cultura", "safavid": "cultura",
    "saffarid": "cultura", "gondar": "cultura", "teotihuacán": "cultura",
    "european": "cultura", "dutch": "cultura", "american colonial": "cultura",
    "Flemish": "cultura",   # discutible — "escuela flamenca" es las dos cosas; n = 1
}
valores = collections.Counter(s for s in style if s)
faltan = set(valores) - set(FAMILIA_STYLE)
assert not faltan, f"valores de style sin clasificar: {faltan}"

familia = np.array([FAMILIA_STYLE[s] if s else None for s in style], dtype=object)
print("LAS TRES FAMILIAS DE `style_title`")
for f in ("movimiento", "período", "cultura"):
    vs = [(v, n) for v, n in valores.most_common() if FAMILIA_STYLE[v] == f]
    print("  %-11s %3d valores · %4d obras · %4.1f %% de las que tienen style"
          % (f, len(vs), sum(n for _, n in vs), 100 * sum(n for _, n in vs) / n_style))
    print("      ", ", ".join(f"{v} ({n})" for v, n in vs[:8]), "…" if len(vs) > 8 else "")

LAS TRES FAMILIAS DE `style_title`
  movimiento   13 valores ·  222 obras · 39.2 % de las que tienen style
       Impressionism (111), Post-Impressionism (32), Realism (26), Renaissance (21), Modernism (9), Folk Art (6), Neoclassicism (5), Baroque (4) …
  período      12 valores ·  253 obras · 44.6 % de las que tienen style
       17th Century (80), 18th Century (38), 16th Century (33), nineteenth century (28), 19th century (25), 15th century (25), sixteenth century (14), 14th century (5) …
  cultura      14 valores ·   92 obras · 16.2 % de las que tienen style
       Japanese (culture or style) (29), Chinese (culture or style) (28), Himalayan (23), safavid (2), teotihuacán (1), gondar (1), Indian (South Asian) (1), european (1) …


## 4 · La pregunta del objetivo opcional de la figura UMAP, con un número y no con una impresión

el objetivo opcional de la figura UMAP pregunta si el espacio separa **tema** o **estilo**. Mirar la figura y decir
*"parece que sí"* no es una respuesta defendible en un oral, así que acá va la misma pregunta
convertida en una medición.

**κ del vecino.** Para cada obra con etiqueta se buscan sus **k = 10 vecinos más cercanos por
coseno**, y se mide con qué frecuencia el vecino lleva **la misma etiqueta**. Ese acuerdo
observado se normaliza igual que el **κ de Cohen** que el proyecto ya usa para los anotadores
(el acuerdo entre anotadores):

$$\kappa = \frac{\text{acuerdo observado} - \text{acuerdo esperado por azar}}{1 - \text{acuerdo esperado por azar}}$$

Tres propiedades que es importante que tenga, porque son las que la vuelven comparable:

- **Se mide en el espacio original de 512 dimensiones, no en la proyección.** UMAP es para
  *mirar*; si la conclusión dependiera de UMAP, dependería de sus hiperparámetros.
- **Normalizar por el azar es obligatorio**, porque los etiquetados tienen distinta cantidad de
  clases: `museo` tiene 2 y `style` 39, así que la pureza cruda favorece al primero y el *lift*
  favorece al segundo. κ corrige las dos cosas.
- **Cada etiquetado se evalúa dentro de su propia población etiquetada**, y por eso la el contrato de salida hace
  además la comparación cabeza a cabeza sobre **las mismas obras**.

In [8]:
def κ_del_vecino(etiquetas, k=K_VECINOS, min_clase=0, X=None):
    """Acuerdo entre la etiqueta de una obra y la de sus k vecinos más cercanos
    por coseno, normalizado como κ. `min_clase` descarta las clases demasiado
    chicas como para poder alcanzar pureza 1 con k vecinos."""
    X = V if X is None else X
    et = list(etiquetas)
    if min_clase:
        c = collections.Counter(e for e in et if e is not None)
        et = [e if (e is not None and c[e] >= min_clase) else None for e in et]
    m = np.array([e is not None for e in et])
    if m.sum() <= k:
        return None
    Xs = X[m]
    y = np.array([e for e in et if e is not None], dtype=object)
    S = Xs @ Xs.T                       # coseno: los vectores tienen norma 1 (la verificación de la norma de los vectores)
    np.fill_diagonal(S, -np.inf)        # una obra no es vecina de sí misma
    vecinos = np.argsort(-S, axis=1)[:, :k]
    mismo = (y[vecinos] == y[:, None])
    obs = float(mismo.mean())
    n = len(y)
    c = collections.Counter(y)
    esp = float(sum((v / n) * ((v - 1) / (n - 1)) for v in c.values()))
    return dict(n=n, n_clases=len(c), k=k, min_clase=min_clase,
                pureza=obs, azar=esp, kappa=(obs - esp) / (1 - esp),
                por_clase={cl: float(mismo[y == cl].mean()) for cl in c},
                n_por_clase=dict(c.most_common()))

def sub(mask, etiquetas):
    return np.array([e if mask[i] else None for i, e in enumerate(etiquetas)], dtype=object)

# los dos etiquetados que no pide la tabla de tokens, costo y la figura, para la §8
PAT_FOLIO = re.compile(r"\bfolios?\b|\bleaf\b|\bleaves\b|\balbum\b|\brecto\b|\bverso\b"
                       r"|\bmanuscript\b|-nama\b|\bsutra\b|\bpurana\b|\bragamala\b")
formato = np.array(["folio de manuscrito" if PAT_FOLIO.search(r.title.lower())
                    else "pintura de caballete" for r in regs], dtype=object)
museo = np.array([r.museum for r in regs], dtype=object)
style_mov = np.array([s if s and FAMILIA_STYLE[s] == "movimiento" else None for s in style], dtype=object)

ETIQUETADOS = [
    ("tema (del título)",        tema),
    ("style (los 39 valores)",   style),
    ("style (sólo movimientos)", style_mov),
    ("culture (top-8 + otras)",  cultura),
    ("formato (folio vs. no)",   formato),
    ("museo",                    museo),
]

print("κ DEL VECINO · espacio CLIP de imagen, 512 dims, coseno, k = %d" % K_VECINOS)
print("%-26s %6s %5s %8s %8s %8s" % ("etiquetado", "n", "clases", "pureza", "azar", "κ"))
print("-" * 68)
resumen = {}
for nombre, et in ETIQUETADOS:
    r = κ_del_vecino(et)
    resumen[nombre] = r
    print("%-26s %6d %5d %8.3f %8.3f %8.3f"
          % (nombre, r["n"], r["n_clases"], r["pureza"], r["azar"], r["kappa"]))

κ DEL VECINO · espacio CLIP de imagen, 512 dims, coseno, k = 10
etiquetado                      n clases   pureza     azar        κ
--------------------------------------------------------------------
tema (del título)             969     6    0.760    0.264    0.674
style (los 39 valores)        567    39    0.402    0.085    0.346
style (sólo movimientos)      222    13    0.587    0.294    0.415


culture (top-8 + otras)      2594     9    0.550    0.168    0.459


formato (folio vs. no)       2605     2    0.894    0.685    0.663


museo                        2605     2    0.742    0.630    0.303


### 4.1 · Por clase — dónde está la señal, y dónde no

La media esconde lo interesante. `style` se separa **exactamente en los valores que no son
estilos**.

In [9]:
for nombre, et, mc in [("tema", tema, 0), ("style (clases con n ≥ 11)", style, 11),
                       ("culture", cultura, 0)]:
    r = κ_del_vecino(et, min_clase=mc)
    print(f"── {nombre}:  κ = {r['kappa']:.3f}")
    for cl, n in sorted(r["n_por_clase"].items(), key=lambda p: -p[1]):
        fam = f"   [{FAMILIA_STYLE[cl]}]" if cl in FAMILIA_STYLE else ""
        print("     %-30s n=%4d  pureza k-NN = %.3f%s" % (cl, n, r["por_clase"][cl], fam))
    print()

── tema:  κ = 0.674
     escena religiosa               n= 383  pureza k-NN = 0.867
     flora y fauna                  n= 201  pureza k-NN = 0.563
     retrato                        n= 175  pureza k-NN = 0.787
     paisaje                        n= 172  pureza k-NN = 0.773
     bodegón                        n=  30  pureza k-NN = 0.610
     desnudo                        n=   8  pureza k-NN = 0.275

── style (clases con n ≥ 11):  κ = 0.380
     Impressionism                  n= 111  pureza k-NN = 0.716   [movimiento]
     17th Century                   n=  80  pureza k-NN = 0.409   [período]
     18th Century                   n=  38  pureza k-NN = 0.211   [período]
     16th Century                   n=  33  pureza k-NN = 0.270   [período]
     Post-Impressionism             n=  32  pureza k-NN = 0.306   [movimiento]
     Japanese (culture or style)    n=  29  pureza k-NN = 0.707   [cultura]
     Chinese (culture or style)     n=  28  pureza k-NN = 0.754   [cultura]
     nineteenth 

── culture:  κ = 0.459
     otras                          n= 788  pureza k-NN = 0.492
     France                         n= 371  pureza k-NN = 0.464
     China                          n= 367  pureza k-NN = 0.731
     Japan                          n= 341  pureza k-NN = 0.674
     Mughal India                   n= 247  pureza k-NN = 0.796
     Italy                          n= 193  pureza k-NN = 0.495
     Netherlands                    n=  99  pureza k-NN = 0.211
     Northern India                 n=  95  pureza k-NN = 0.378
     United States                  n=  93  pureza k-NN = 0.218



### 4.2 · Cabeza a cabeza — tema vs. `style` sobre **las mismas obras**

Comparar κ entre poblaciones distintas deja la puerta abierta a *"es que el tema se midió sobre
otras obras"*. Las obras que tienen **las dos** etiquetas son pocas —`style` sólo existe en AIC—
pero son la comparación limpia, y además **controlan el museo**, porque son todas del mismo.

In [10]:
ambas = np.array([t is not None and s is not None for t, s in zip(tema, style)])
print("obras con tema Y style: %d  (museos: %s)"
      % (ambas.sum(), dict(collections.Counter(r.museum for r, m in zip(regs, ambas) if m))))
print()
print("%-10s %-14s %5s %6s %8s %8s %8s" % ("", "", "n", "clases", "pureza", "azar", "κ"))
for et_nom, et in [("tema", tema), ("style", style)]:
    for mc, etq in [(0, "todas las clases"), (11, "clases con n ≥ 11")]:
        r = κ_del_vecino(sub(ambas, et), min_clase=mc)
        print("%-10s %-14s %5d %6d %8.3f %8.3f %8.3f"
              % (et_nom, etq, r["n"], r["n_clases"], r["pureza"], r["azar"], r["kappa"]))

obras con tema Y style: 189  (museos: {'aic': 189})

                              n clases   pureza     azar        κ
tema       todas las clases   189      6    0.775    0.296    0.680
tema       clases con n ≥ 11   185      5    0.788    0.309    0.693
style      todas las clases   189     22    0.290    0.085    0.224
style      clases con n ≥ 11   119      6    0.424    0.190    0.289


### 4.3 · Controles — ¿el tema es otra cosa disfrazada?

Un κ alto para el tema no sirve si el tema está confundido con la cultura (los paisajes son casi
todos chinos y japoneses) o con el formato. Tres controles, que son lo que vuelve la conclusión
defendible:

1. **tema dentro de cada formato** — ¿el tema sigue separando si se mira sólo folios, o sólo
   pinturas de caballete?
2. **tema dentro de cada cultura** — el control decisivo: si el tema separa *adentro* de cada
   tradición, no es la tradición disfrazada.
3. **formato dentro de cada cultura** — el mismo control, para el hallazgo no pedido de la §8.

In [11]:
print("CONTROL 1 · tema DENTRO de cada formato")
for f in ("folio de manuscrito", "pintura de caballete"):
    m = (formato == f)
    r = κ_del_vecino(sub(m, tema))
    print("   %-22s n=%4d clases=%d  κ = %.3f" % (f, r["n"], r["n_clases"], r["kappa"]))

print("\nCONTROL 2 · tema DENTRO de cada cultura del top-8")
for c in TOP8:
    m = (cultura == c)
    r = κ_del_vecino(sub(m, tema))
    if r:
        flag = "   ⚠️ n chico" if r["n"] < 40 else ""
        print("   %-16s n=%4d clases=%d  κ = %.3f%s" % (c, r["n"], r["n_clases"], r["kappa"], flag))

print("\nCONTROL 3 · formato DENTRO de cada cultura  (sólo donde hay masa de los dos)")
for c in TOP8 + ["otras"]:
    m = (cultura == c)
    nf = collections.Counter(formato[m])
    a, b = nf["folio de manuscrito"], nf["pintura de caballete"]
    if min(a, b) < 20:
        print("   %-16s — un solo formato con masa (folio %d / caballete %d): no es comparable" % (c, a, b))
        continue
    r = κ_del_vecino(sub(m, formato))
    print("   %-16s n=%4d (folio %3d / caballete %3d)  κ = %.3f" % (c, r["n"], a, b, r["kappa"]))

CONTROL 1 · tema DENTRO de cada formato
   folio de manuscrito    n= 162 clases=4  κ = 0.629
   pintura de caballete   n= 807 clases=6  κ = 0.681

CONTROL 2 · tema DENTRO de cada cultura del top-8
   France           n=  92 clases=6  κ = 0.596


   China            n= 176 clases=4  κ = 0.598


   Japan            n= 137 clases=4  κ = 0.497
   Mughal India     n=  53 clases=4  κ = 0.321
   Italy            n= 114 clases=4  κ = 0.702
   Netherlands      n=  47 clases=5  κ = 0.655
   Northern India   n=  32 clases=4  κ = 0.051   ⚠️ n chico
   United States    n=  16 clases=6  κ = 0.103   ⚠️ n chico

CONTROL 3 · formato DENTRO de cada cultura  (sólo donde hay masa de los dos)
   France           — un solo formato con masa (folio 6 / caballete 365): no es comparable


   China            n= 367 (folio  93 / caballete 274)  κ = 0.541
   Japan            — un solo formato con masa (folio 1 / caballete 340): no es comparable
   Mughal India     n= 247 (folio 218 / caballete  29)  κ = 0.490
   Italy            — un solo formato con masa (folio 0 / caballete 193): no es comparable
   Netherlands      — un solo formato con masa (folio 1 / caballete 98): no es comparable
   Northern India   n=  95 (folio  20 / caballete  75)  κ = 0.016
   United States    — un solo formato con masa (folio 0 / caballete 93): no es comparable
   otras            n= 788 (folio 168 / caballete 620)  κ = 0.524


## 5 · La proyección

`metric="cosine"` declarado y `random_state=7369` fijado. Con la semilla fija, dos corridas dan
**exactamente** el mismo arreglo, que es lo que vuelve la figura citable.

In [12]:
import warnings
warnings.filterwarnings("ignore")        # umap-learn avisa que random_state desactiva el paralelismo
import umap

reductor = umap.UMAP(**UMAP_PARAMS)
P = reductor.fit_transform(V.astype(np.float32))
print("proyección:", P.shape, "· parámetros:", UMAP_PARAMS)
print("x ∈ [%.2f, %.2f] · y ∈ [%.2f, %.2f]" % (P[:, 0].min(), P[:, 0].max(), P[:, 1].min(), P[:, 1].max()))

proyección: (2605, 2) · parámetros: {'n_components': 2, 'n_neighbors': 15, 'min_dist': 0.1, 'metric': 'cosine', 'random_state': 7369}
x ∈ [-4.99, 16.19] · y ∈ [-0.48, 12.50]


### ¿La figura muestra lo que el número dice?

UMAP distorsiona: podría inventar estructura que en 512 dimensiones no está, o destruir la que
sí está. La forma barata de verificarlo es **recalcular el mismo κ sobre las coordenadas 2-D** y
comparar. Si el orden de los etiquetados se conserva, la figura es una lectura fiel del espacio
y no un artefacto de la proyección.

In [13]:
print("%-26s %10s %10s" % ("etiquetado", "κ en 512-d", "κ en 2-d"))
print("-" * 48)
P64 = P.astype(np.float64)

def κ_2d(etiquetas, k=K_VECINOS):
    m = np.array([e is not None for e in etiquetas])
    X = P64[m]
    y = np.array([e for e in etiquetas if e is not None], dtype=object)
    D = ((X[:, None, :] - X[None, :, :]) ** 2).sum(-1)
    np.fill_diagonal(D, np.inf)
    vecinos = np.argsort(D, axis=1)[:, :k]
    obs = float((y[vecinos] == y[:, None]).mean())
    n = len(y); c = collections.Counter(y)
    esp = float(sum((v / n) * ((v - 1) / (n - 1)) for v in c.values()))
    return (obs - esp) / (1 - esp)

κ_2d_por_etiquetado = {}
for nombre, et in ETIQUETADOS:
    k2 = κ_2d(et)
    κ_2d_por_etiquetado[nombre] = k2
    print("%-26s %10.3f %10.3f" % (nombre, resumen[nombre]["kappa"], k2))

etiquetado                 κ en 512-d   κ en 2-d
------------------------------------------------
tema (del título)               0.674      0.642
style (los 39 valores)          0.346      0.320
style (sólo movimientos)        0.415      0.347


culture (top-8 + otras)         0.459      0.441


formato (folio vs. no)          0.663      0.614


museo                           0.303      0.264


## 6 · Las figuras

**Cómo están construidas, y por qué así.** Un scatter de 2.605 puntos con diez colores a la vez
no se lee: los puntos se tapan y los tonos se confunden. La forma correcta para *"¿este
subconjunto se concentra?"* es **resaltar uno y agrisar el resto**, en **pequeños múltiplos** —
un panel por clase, todos con la misma proyección y los mismos ejes. Así cada panel responde su
propia pregunta y la comparación entre paneles es visual y directa.

La paleta es de un solo tono por panel sobre gris de contexto, con el nombre de la clase
**escrito en el panel** (la identidad nunca va sólo en el color), y los ejes de UMAP van sin
números porque **sus unidades no significan nada**.

In [14]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# ── Paleta ──────────────────────────────────────────────────────────────────
SUP    = "#fcfcfb"      # superficie clara (la figura va impresa en un informe)
TINTA  = "#0b0b0b"      # tinta primaria
TINTA2 = "#52514e"      # tinta secundaria
MUDO   = "#898781"      # etiquetas de eje
GRIS   = "#c3c2b7"      # el gris de de-énfasis: el resto del corpus
MARCO  = "#e1e0d9"
AZUL, NARANJA, AGUA = "#2a78d6", "#eb6834", "#1baf7a"   # slots 1–3, validados a todo-par
RAMPA = LinearSegmentedColormap.from_list("azul", [
    "#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7", "#3987e5",
    "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"])

plt.rcParams.update({
    "figure.facecolor": SUP, "axes.facecolor": SUP, "savefig.facecolor": SUP,
    "font.family": "sans-serif", "font.size": 8,
    "axes.edgecolor": MARCO, "axes.linewidth": 0.6,
    "text.color": TINTA, "axes.labelcolor": TINTA2,
    "xtick.color": MUDO, "ytick.color": MUDO, "figure.dpi": 110,
})
PIE = ("corpus indexado del split %s · %d obras · CLIP %s (preproc %s) · "
       "UMAP n_neighbors=%d min_dist=%s metric=%s random_state=%d"
       % (SPLIT_ID, n_tot, CHECKPOINT, PREPROC, UMAP_PARAMS["n_neighbors"],
          UMAP_PARAMS["min_dist"], UMAP_PARAMS["metric"], SEMILLA))

def num(x, d=2):
    """Coma decimal: el resto del proyecto escribe `99,97 %`, no `99.97 %`."""
    return ("%.*f" % (d, x)).replace(".", ",")

# Límites comunes a TODOS los paneles. Es lo que vuelve comparables los
# pequeños múltiplos: si cada panel se autoescalara, dos nubes con distinta
# forma se verían iguales.
MARGEN = 0.03
_dx, _dy = np.ptp(P[:, 0]), np.ptp(P[:, 1])
XLIM = (P[:, 0].min() - MARGEN * _dx, P[:, 0].max() + MARGEN * _dx)
YLIM = (P[:, 1].min() - MARGEN * _dy, P[:, 1].max() + MARGEN * _dy)

def _marco(ax):
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlim(*XLIM); ax.set_ylim(*YLIM)
    for s in ax.spines.values():
        s.set_color(MARCO); s.set_linewidth(0.6)

def panel(ax, mask, color, título, sub):
    """Un panel de énfasis: la clase en color, el resto del corpus en gris."""
    ax.scatter(P[~mask, 0], P[~mask, 1], s=2.8, c=GRIS, alpha=0.45, linewidths=0, rasterized=True)
    ax.scatter(P[mask, 0], P[mask, 1], s=7.5, c=color, alpha=0.95,
               linewidths=0.25, edgecolors=SUP, rasterized=True)
    ax.set_title(título, fontsize=8.5, color=TINTA, loc="left", pad=14)
    ax.text(0.0, 1.015, sub, transform=ax.transAxes, fontsize=6.8, color=MUDO,
            ha="left", va="bottom")
    _marco(ax)

def encabezar(fig, título, bajada, pie=PIE):
    fig.suptitle(título, fontsize=10.5, x=0.009, ha="left", y=0.988, color=TINTA)
    fig.text(0.009, 0.948, bajada, fontsize=7.2, color=TINTA2, ha="left", va="top")
    fig.text(0.009, 0.012, pie, fontsize=6.2, color=MUDO, ha="left", va="bottom")

def guardar(fig, nombre):
    SALIDA.mkdir(parents=True, exist_ok=True)
    ruta = SALIDA / nombre
    fig.savefig(ruta, dpi=200, bbox_inches="tight", facecolor=SUP)
    print("escrito:", ruta.relative_to(RAIZ))
    return ruta

rutas = {}

### Figura 1 — **tema**

In [15]:
r_tema = κ_del_vecino(tema)
fig, axes = plt.subplots(2, 4, figsize=(12.6, 6.6))
for ax, t in zip(axes.flat, TEMAS):
    m = (tema == t)
    panel(ax, m, AZUL, t, "n = %d · %s %% del corpus · pureza k-NN %s"
          % (m.sum(), num(100 * m.sum() / n_tot, 1), num(r_tema["por_clase"][t])))
m = np.array([t is None for t in tema])
panel(axes.flat[len(TEMAS)], m, GRIS, "sin etiquetar por el léxico",
      "n = %d · %s %% del corpus" % (m.sum(), num(100 * m.sum() / n_tot, 1)))
for ax in axes.flat[len(TEMAS) + 1:]:
    ax.axis("off")
encabezar(fig,
    "UMAP del espacio CLIP de imagen · coloreado por TEMA",
    "El tema NO es un campo: se deriva del `title` escrito por el museo (los tres ciclos de vida), nunca de CLIP.  "
    "Cada panel resalta un tema sobre la nube completa.\n"
    "Cobertura del etiquetado: %s %% de las %s obras.   κ del vecino = %s en 512 dims."
    % (num(100 * n_tema / n_tot, 1), f"{n_tot:,}".replace(",", "."), num(r_tema["kappa"], 3)))
fig.tight_layout(rect=(0, 0.02, 1, 0.925))
rutas["tema"] = guardar(fig, "umap-01-tema.png")
plt.show()

escrito: artifacts/informes/umap-01-tema.png


### Figura 2 — **`culture`**

In [16]:
r_cult = κ_del_vecino(cultura)
fig, axes = plt.subplots(2, 5, figsize=(15.2, 6.6))
for ax, c in zip(axes.flat, CLASES_CULTURA):
    m = (cultura == c)
    panel(ax, m, AZUL, c, "n = %d · %s %% · pureza k-NN %s"
          % (m.sum(), num(100 * m.sum() / n_tot, 1), num(r_cult["por_clase"][c])))
m = np.array([c is None for c in cultura])
panel(axes.flat[len(CLASES_CULTURA)], m, GRIS, "sin valor de culture",
      "n = %d · %s %% del corpus" % (m.sum(), num(100 * m.sum() / n_tot, 1)))
for ax in axes.flat[len(CLASES_CULTURA) + 1:]:
    ax.axis("off")
encabezar(fig,
    "UMAP del espacio CLIP de imagen · coloreado por CULTURE",
    "Criterio de normalización declarado: `%s` (config/default.toml).  "
    "%d etiquetas distintas en el corpus → se colorean las top-8 por n, el resto en `otras`.\n"
    "ATENCIÓN: este agrupamiento es un dato declarado de la figura y NO cierra el criterio de normalización de culture: "
    "China/Chinese, Netherlands/Netherlandish y la India en once etiquetas siguen sin unirse.   κ = %s"
    % (CRITERIO_CULTURE, len(conteo_cult), num(r_cult["kappa"], 3)))
fig.tight_layout(rect=(0, 0.02, 1, 0.918))
rutas["culture"] = guardar(fig, "umap-02-culture.png")
plt.show()

escrito: artifacts/informes/umap-02-culture.png


### Figura 3 — **`year`**

In [17]:
hay = ~np.isnan(year)

# La rampa se recorta al percentil 1 por una razón declarada: 9 obras son
# anteriores al año 1000 —la más antigua, un ostracon egipcio del −1295— y sin
# recortar se llevan el 60 % del gradiente, dejando los 2.595 restantes en dos
# tonos indistinguibles. Las 9 se dibujan con el extremo claro y la barra lo
# marca con una flecha; sus ids van en el sidecar. El panel de la derecha, que
# usa los tramos del split, no necesita recorte y es la lectura robusta.
VMIN, VMAX = float(np.percentile(year[hay], 1)), float(np.nanmax(year))
n_recortadas = int((year[hay] < VMIN).sum())

fig = plt.figure(figsize=(15.2, 6.4))
gs = fig.add_gridspec(2, 5, width_ratios=[1.35, 0.085, 1, 1, 1], wspace=0.17, hspace=0.34,
                      left=0.035, right=0.995, top=0.845, bottom=0.055)

# ── izquierda: la escala continua ──────────────────────────────────────────
ax = fig.add_subplot(gs[:, 0])
ax.scatter(P[~hay, 0], P[~hay, 1], s=11, c=GRIS, alpha=0.95, linewidths=0)
sc = ax.scatter(P[hay, 0], P[hay, 1], s=4.5, c=np.clip(year[hay], VMIN, VMAX), cmap=RAMPA,
                vmin=VMIN, vmax=VMAX, linewidths=0, rasterized=True)
# pad mayor que el de `panel()`: acá la bajada del panel son dos líneas
ax.set_title("year, escala secuencial de un solo tono", fontsize=8.5, color=TINTA, loc="left", pad=26)
ax.text(0.0, 1.015,
        "con valor: %d / %d = %s %% · sin valor (gris): %s\nrampa recortada en p1 = %d — %d obras más antiguas, hasta el año %d"
        % (hay.sum(), n_tot, num(100 * hay.sum() / n_tot), 
           ", ".join(r.artwork_id for r, y in zip(regs, year) if np.isnan(y)),
           VMIN, n_recortadas, np.nanmin(year)),
        transform=ax.transAxes, fontsize=6.8, color=MUDO, ha="left", va="bottom")
_marco(ax)
cax = fig.add_subplot(gs[:, 1])
cb = fig.colorbar(sc, cax=cax, extend="min")
cb.outline.set_edgecolor(MARCO); cb.outline.set_linewidth(0.6)
cb.ax.tick_params(labelsize=6.5, color=MUDO, labelcolor=MUDO)
cb.ax.set_title("año", fontsize=7, color=TINTA2, pad=5)

# ── derecha: los seis tramos, como pequeños múltiplos ──────────────────────
#  Seis pasos de un mismo tono no se distinguen en un scatter superpuesto, así
#  que cada tramo va en su propio panel. Los tramos son los de
#  `config/default.toml § [split.estrato]`, no unos inventados para la figura.
TRAMOS = [("hasta 1399", -99999, 1399), ("1400–1599", 1400, 1599), ("1600–1699", 1600, 1699),
          ("1700–1799", 1700, 1799), ("1800–1874", 1800, 1874), ("1875–1949", 1875, 99999)]
pasos = [RAMPA(v) for v in np.linspace(0.22, 0.98, len(TRAMOS))]
for i, ((nom, a_, b_), col) in enumerate(zip(TRAMOS, pasos)):
    ax = fig.add_subplot(gs[i // 3, 2 + i % 3])
    m = hay & (year >= a_) & (year <= b_)
    panel(ax, m, col, nom, "n = %d · %s %% del corpus" % (m.sum(), num(100 * m.sum() / n_tot, 1)))

encabezar(fig,
    "UMAP del espacio CLIP de imagen · coloreado por YEAR",
    "`year` es continuo, así que la escala es secuencial de un solo tono claro→oscuro, no categórica "
    "y no un arcoíris.\nCobertura %s %%: falta una sola obra de las %s.   "
    "A la derecha, los seis tramos de período del split, un panel por tramo."
    % (num(100 * hay.sum() / n_tot), f"{n_tot:,}".replace(",", ".")))
rutas["year"] = guardar(fig, "umap-03-year.png")
plt.show()

escrito: artifacts/informes/umap-03-year.png


### Figura 4 — **`style`**, el contraste

In [18]:
TOPE_STYLE = 20
vals = [v for v, n in valores.most_common() if n >= TOPE_STYLE]
r_sty = κ_del_vecino(style, min_clase=11)
COLOR_FAM = {"movimiento": AZUL, "período": NARANJA, "cultura": AGUA}

filas = int(np.ceil((len(vals) + 1) / 5))
fig, axes = plt.subplots(filas, 5, figsize=(15.2, 3.3 * filas))
for ax, v in zip(axes.flat, vals):
    m = (style == v)
    fam = FAMILIA_STYLE[v]
    panel(ax, m, COLOR_FAM[fam], "%s" % v,
          "[%s] · n = %d · pureza k-NN %s" % (fam, m.sum(), num(r_sty["por_clase"].get(v, float("nan")))))
m = np.array([s is None for s in style])
panel(axes.flat[len(vals)], m, GRIS, "sin valor de style",
      "n = %d · %s %% del corpus" % (m.sum(), num(100 * m.sum() / n_tot, 1)))
for ax in axes.flat[len(vals) + 1:]:
    ax.axis("off")
r_sty_todo = κ_del_vecino(style)
r_mov = κ_del_vecino(style_mov)
encabezar(fig,
    "UMAP del espacio CLIP de imagen · coloreado por STYLE  (el contraste)",
    "`style` sólo existe en AIC: %d / %d obras del corpus = %s %%.  Se muestran los %d valores con n ≥ %d.\n"
    "El color del panel es la FAMILIA del valor (ver la referencia arriba a la derecha), "
    "porque la mayoría de los valores de `style_title` no son estilos (la descomposición en componentes).\n"
    "κ del vecino: %s con los %d valores · %s con sólo los movimientos."
    % (n_style, n_tot, num(100 * n_style / n_tot, 1), len(vals), TOPE_STYLE,
       num(r_sty_todo["kappa"], 3), r_sty_todo["n_clases"], num(r_mov["kappa"], 3)))
from matplotlib.patches import Patch
leg = fig.legend(handles=[Patch(facecolor=COLOR_FAM[f], edgecolor="none",
                                label="%s  (%d valores, %d obras)"
                                      % (f, sum(1 for v in valores if FAMILIA_STYLE[v] == f),
                                         sum(n for v, n in valores.items() if FAMILIA_STYLE[v] == f)))
                          for f in ("movimiento", "período", "cultura")],
                 loc="upper right", bbox_to_anchor=(0.995, 0.995), fontsize=7.2,
                 frameon=True, facecolor=SUP, edgecolor=MARCO, labelcolor=TINTA2,
                 title="familia del valor de `style_title`", title_fontsize=7.2,
                 handlelength=1.1, handleheight=1.1, borderpad=0.7)
leg.get_frame().set_linewidth(0.6)
leg.get_title().set_color(TINTA)
fig.tight_layout(rect=(0, 0.02, 1, 1 - 0.30 / filas))
rutas["style"] = guardar(fig, "umap-04-style.png")
plt.show()

escrito: artifacts/informes/umap-04-style.png


### Figura 5 — la respuesta al objetivo opcional de la figura UMAP en un solo gráfico

El κ de cada etiquetado, ordenado. Es el gráfico que contesta *"¿separa temas? ¿separa
estilos?"* sin que haya que interpretar una nube de puntos.

In [19]:
orden = sorted(ETIQUETADOS, key=lambda p: resumen[p[0]]["kappa"])
nombres = [n for n, _ in orden]
ks = [resumen[n]["kappa"] for n in nombres]
col = [AZUL if "tema" in n else (NARANJA if "style" in n else GRIS) for n in nombres]

fig, ax = plt.subplots(figsize=(9.4, 3.9))
barras = ax.barh(nombres, ks, color=col, height=0.62, linewidth=0)
for b, n in zip(barras, nombres):
    r = resumen[n]
    ax.text(b.get_width() + 0.012, b.get_y() + b.get_height() / 2,
            "κ = %s        n = %d · %d clases" % (num(r["kappa"], 3), r["n"], r["n_clases"]),
            va="center", fontsize=7.4, color=TINTA2)
ax.set_xlim(0, 1.0)
ax.set_xlabel("κ del vecino — acuerdo con los %d vecinos más cercanos, corregido por azar" % K_VECINOS,
              fontsize=7.4)
ax.tick_params(axis="y", length=0, labelsize=8, labelcolor=TINTA)
ax.tick_params(axis="x", labelsize=7)
ax.grid(axis="x", color=MARCO, linewidth=0.6, alpha=0.9)
ax.set_axisbelow(True)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color(MARCO)
encabezar(fig,
    "¿El espacio CLIP separa TEMA o separa ESTILO?  — el objetivo opcional de la figura UMAP",
    "Medido en el espacio original de 512 dimensiones, NO en la proyección: UMAP es para mirar, "
    "no para concluir.\nκ = (acuerdo observado − acuerdo por azar) / (1 − acuerdo por azar), "
    "la misma normalización que el κ de Cohen del acuerdo entre anotadores.",
    pie="corpus indexado del split %s · %d obras · CLIP %s (preproc %s) · "
        "medido en el espacio de 512 dims: los parámetros de UMAP no intervienen en este gráfico"
        % (SPLIT_ID, n_tot, CHECKPOINT, PREPROC))
fig.tight_layout(rect=(0, 0.03, 1, 0.84))
rutas["separacion"] = guardar(fig, "umap-05-separacion.png")
plt.show()

escrito: artifacts/informes/umap-05-separacion.png


### Figura 6 — los dos hallazgos que **nadie pidió**

`el registro de decisiones el criterio de idioma` encarga cuatro coloreados. Estos dos no estaban en la lista y salieron de mirar la
estructura macro de la proyección: el corpus se parte en dos bloques grandes que **no son ni
tema ni estilo**.

In [20]:
fig, axes = plt.subplots(1, 4, figsize=(15.2, 4.1))
r_fmt, r_mus = resumen["formato (folio vs. no)"], resumen["museo"]
for ax, (et, cl, col, r) in zip(axes.flat, [
        (formato, "folio de manuscrito",  AZUL,    r_fmt),
        (formato, "pintura de caballete", NARANJA, r_fmt),
        (museo,   "aic",                  AZUL,    r_mus),
        (museo,   "cma",                  NARANJA, r_mus)]):
    m = (et == cl)
    panel(ax, m, col, cl, "n = %d · %s %% · pureza k-NN %s"
          % (m.sum(), num(100 * m.sum() / n_tot, 1), num(r["por_clase"][cl])))
encabezar(fig,
    "Dos etiquetados que `el registro de decisiones el criterio de idioma` no pedía",
    "FORMATO (κ = %s) — folio de álbum o manuscrito vs. pintura de caballete, derivado del título. "
    "Es el segundo etiquetado más separado, por encima de `culture` y muy por encima de `style`.\n"
    "MUSEO (κ = %s) — el más bajo de los seis: los dos museos NO forman clusters propios, "
    "así que la estructura no es un artefacto de procedencia."
    % (num(r_fmt["kappa"], 3), num(r_mus["kappa"], 3)))
fig.tight_layout(rect=(0, 0.02, 1, 0.875))
rutas["hallazgos"] = guardar(fig, "umap-06-hallazgos-no-pedidos.png")
plt.show()

escrito: artifacts/informes/umap-06-hallazgos-no-pedidos.png


## 7 · El sidecar de procedencia

`el registro de decisiones el criterio de idioma` regla 2: *"la figura declara su procedencia, en el pie o en un sidecar:
`split_id`, el checkpoint de CLIP, los parámetros de UMAP y la semilla"*. Van las dos cosas: el
pie está impreso en cada PNG y el sidecar lleva además **todas las cifras que este notebook
publica**, para que ninguna tenga que transcribirse a mano.

> 📌 **Es la lección de ` § Quinta vez` aplicada acá:** *"una cifra en `docs/` lleva el
> artefacto del que salió"*. Las seis veces que el proyecto publicó una cifra equivocada, el
> error fue de alguien transcribiendo; **ninguna fue del código.** Este sidecar existe para que
> nadie tenga que transcribir.

In [21]:
def _limpio(r):
    return {k: r[k] for k in ("n", "n_clases", "k", "pureza", "azar", "kappa", "por_clase", "n_por_clase")}

sidecar = {
    "artefacto": "FiguraUMAP",
    "para": ["la tabla de tokens, costo y la figura", "el objetivo opcional de la figura UMAP", "el registro de decisiones el criterio de idioma"],
    "procedencia": {
        "split_id": SPLIT_ID,
        "catalogo_id": CATALOGO_ID,
        "checkpoint": CHECKPOINT,
        "version_del_preprocesado": PREPROC,
        "semilla": SEMILLA,
        "umap": {k: v for k, v in UMAP_PARAMS.items()},
        "k_vecinos": K_VECINOS,
        "fuente_de_los_vectores": "caché de embeddings de imagen (data/embeddings-imagen/), no Chroma",
        "variante_id": None,
        "nota_variante": (
            "el módulo de evaluación dice que la figura es sobre el conjunto indexado de UNA VARIANTE y "
            "que el criterio de `culture` se lee de la receta de la variante. No hay variante "
            "registrada (espera la API key), así que la membresía se lee del ManifiestoDeSplit —que "
            "es su dueño, la regla de qué se versiona— y el criterio de `culture` de config/default.toml. La proyección no "
            "depende de la variante: el caché está indexado por (obra, hash de bytes, checkpoint, "
            "versión del preprocesado). Lo único a rehacer cuando la variante exista es el panel de "
            "`culture`, y sólo si la receta declara otro criterio."
        ),
        "python": platform.python_version(),
        "umap_learn": umap.__version__,
        "numpy": np.__version__,
        "matplotlib": matplotlib.__version__,
    },
    "poblacion": {
        "nombre": "corpus indexado del split " + SPLIT_ID,
        "n": n_tot,
        "no_es": {"universo_normalizado": 5915, "universo_con_imagen_embebido": 5751,
                  "elegibles_con_texto_e_imagen": 2894, "held_out": 289},
        "nota": ("la teoría aplicada § UMAP, el registro de decisiones el criterio de idioma y la tabla de tokens, costo y la figura dicen los tres que la figura es sobre "
                 "las INDEXADAS. Las 5.751 embebidas serían otra figura y dirían otra cosa."),
        "verificado": {"dim": int(V.shape[1]), "norma_min": float(normas.min()),
                       "norma_max": float(normas.max()), "la verificación de la norma de los vectores": "‖v‖ = 1"},
    },
    "coloreados": {
        "tema": {
            "origen": "derivado del campo `title` del museo, con regex declaradas",
            "por_que_no_CLIP": ("etiquetar el tema con CLIP zero-shot sería circular: etiquetaría los "
                                "datos con el mismo espacio que la figura evalúa, y los clusters "
                                "coincidirían con las etiquetas por construcción"),
            "lexico": {n: p for n, p in LÉXICO_TEMA},
            "precedencia": "el orden del léxico; gana el primer patrón que machea",
            "cobertura": {"n_etiquetadas": n_tema, "pct": round(100 * n_tema / n_tot, 1),
                          "n_sin_etiquetar": n_tot - n_tema},
            "n_por_clase": {t: int((tema == t).sum()) for t in TEMAS},
        },
        "culture": {
            "criterio_de_normalizacion": CRITERIO_CULTURE,
            "fuente_del_criterio": "config/default.toml § [normalizacion.culture]",
            "n_etiquetas_en_el_corpus": len(conteo_cult),
            "agrupamiento_declarado": "top-8 por n + `otras`; sin valor → gris",
            "top8": {e: conteo_cult[e] for e in TOP8},
            "n_otras": int((cultura == "otras").sum()),
            "n_sin_culture": n_sin_cult,
            "cobertura_pct": round(100 * (n_tot - n_sin_cult) / n_tot, 2),
            "no_cierra": ("el criterio de normalización de culture. Es un corte por frecuencia, no una tabla de sinónimos: "
                          "China/Chinese, Netherlands/Netherlandish y la India en once etiquetas "
                          "siguen sin unirse."),
        },
        "year": {
            "escala": "secuencial de un solo tono, claro→oscuro",
            "n_con_valor": int(hay.sum()), "cobertura_pct": round(100 * hay.sum() / n_tot, 2),
            "sin_valor": [r.artwork_id for r, y in zip(regs, year) if np.isnan(y)],
            "rango": [int(np.nanmin(year)), int(np.nanmax(year))],
            "rampa_recortada_en": [VMIN, VMAX],
            "nota_recorte": ("la rampa se recorta en el percentil 1 porque 9 obras anteriores al año "
                             "1000 se llevarían el 60 % del gradiente; el panel de tramos no se recorta"),
            "anteriores_al_1000": {r.artwork_id: int(r.year) for r in regs
                                   if r.year is not None and r.year < 1000},
        },
        "style": {
            "n_con_valor": n_style, "cobertura_pct": round(100 * n_style / n_tot, 1),
            "por_museo": {m: round(100 * sum(1 for r, s in zip(regs, style) if r.museum == m and s)
                                   / sum(1 for r in regs if r.museum == m), 1) for m in ("aic", "cma")},
            "n_valores_distintos": len(valores),
            "familias_declaradas": {f: {"n_valores": sum(1 for v in valores if FAMILIA_STYLE[v] == f),
                                        "n_obras": sum(n for v, n in valores.items() if FAMILIA_STYLE[v] == f)}
                                    for f in ("movimiento", "período", "cultura")},
            "clasificacion": FAMILIA_STYLE,
        },
    },
    "separacion": {
        "metrica": ("κ del vecino: acuerdo entre la etiqueta de una obra y la de sus k vecinos más "
                    "cercanos por coseno, normalizado como el κ de Cohen. Medido en el espacio "
                    "original de 512 dims, no en la proyección."),
        "k": K_VECINOS,
        "por_etiquetado": {n: _limpio(resumen[n]) for n, _ in ETIQUETADOS},
        "kappa_en_2d_de_umap": κ_2d_por_etiquetado,
        "cabeza_a_cabeza": {
            "poblacion": "las %d obras que tienen tema Y style (todas de AIC)" % int(ambas.sum()),
            "tema": _limpio(κ_del_vecino(sub(ambas, tema))),
            "style": _limpio(κ_del_vecino(sub(ambas, style))),
        },
        "controles": {
            "tema_dentro_de_cada_formato": {
                f: _limpio(κ_del_vecino(sub(formato == f, tema)))
                for f in ("folio de manuscrito", "pintura de caballete")},
            "tema_dentro_de_cada_cultura": {
                c: _limpio(κ_del_vecino(sub(cultura == c, tema))) for c in TOP8
                if κ_del_vecino(sub(cultura == c, tema))},
        },
    },
    "figuras": {k: str(v.relative_to(RAIZ)) for k, v in rutas.items()},
}
ruta_sidecar = SALIDA / "umap-procedencia.json"
ruta_sidecar.write_text(json.dumps(sidecar, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("escrito:", ruta_sidecar.relative_to(RAIZ))
print(json.dumps(sidecar["separacion"]["por_etiquetado"], indent=1, ensure_ascii=False)[:900], "…")

escrito: artifacts/informes/umap-procedencia.json
{
 "tema (del título)": {
  "n": 969,
  "n_clases": 6,
  "k": 10,
  "pureza": 0.760061919504644,
  "azar": 0.2636419073936665,
  "kappa": 0.6741557091521909,
  "por_clase": {
   "escena religiosa": 0.8671018276762402,
   "retrato": 0.7874285714285715,
   "flora y fauna": 0.56318407960199,
   "paisaje": 0.7726744186046511,
   "bodegón": 0.61,
   "desnudo": 0.275
  },
  "n_por_clase": {
   "escena religiosa": 383,
   "flora y fauna": 201,
   "retrato": 175,
   "paisaje": 172,
   "bodegón": 30,
   "desnudo": 8
  }
 },
 "style (los 39 valores)": {
  "n": 567,
  "n_clases": 39,
  "k": 10,
  "pureza": 0.4015873015873016,
  "azar": 0.08547871445397948,
  "kappa": 0.34565470714504754,
  "por_clase": {
   "18th Century": 0.19736842105263158,
   "16th Century": 0.2636363636363636,
   "19th century": 0.064,
   "Impressionism": 0.6972972972972973,
   "Himalayan": 0.8173913043478261,
   "sixteenth ce …


## 8 · La lectura de la figura

> 📌 **Las cifras de esta sección las emite la celda de abajo, no están transcritas.** Es la
> lección de ` § Quinta vez` aplicada: el proyecto publicó **seis** cifras equivocadas
> y **las seis fueron de alguien transcribiendo, ninguna del código**. Una conclusión escrita a
> mano con números pegados a mano es el séptimo caso esperando a pasar.

In [22]:
from IPython.display import Markdown, display

r_t, r_s = resumen["tema (del título)"], resumen["style (los 39 valores)"]
r_mov, r_c = resumen["style (sólo movimientos)"], resumen["culture (top-8 + otras)"]
r_f, r_m = resumen["formato (folio vs. no)"], resumen["museo"]
h2h_t = κ_del_vecino(sub(ambas, tema))
h2h_s = κ_del_vecino(sub(ambas, style))

def num(x, d=2):
    """Coma decimal: el resto del proyecto escribe `99,97 %`, no `99.97 %`."""
    return ("%.*f" % (d, x)).replace(".", ",")

ctrl_cult = {c: κ_del_vecino(sub(cultura == c, tema)) for c in TOP8}
ctrl_cult = {c: r for c, r in ctrl_cult.items() if r and r["n"] >= 40}
cult_txt = " · ".join("%s %s" % (c, num(r["kappa"]))
                      for c, r in sorted(ctrl_cult.items(), key=lambda p: -p[1]["kappa"]))
paisajes = collections.Counter(c for c, t in zip(cultura, tema) if t == "paisaje")
pais_txt = ", ".join("%s (%d)" % (c, n) for c, n in paisajes.most_common() if c != "otras")

fmt_cult = {}
for c in TOP8 + ["otras"]:
    m = (cultura == c)
    nf = collections.Counter(formato[m])
    if min(nf["folio de manuscrito"], nf["pintura de caballete"]) >= 20:
        fmt_cult[c] = (κ_del_vecino(sub(m, formato))["kappa"], nf["folio de manuscrito"])
# Se dicen las que sostienen y las que no: una lista que mezcla 0,54 con 0,02
# bajo la palabra "se sostiene" se contradice sola.
sostienen = {c: v for c, v in fmt_cult.items() if v[0] >= 0.30}
no_sost   = {c: v for c, v in fmt_cult.items() if v[0] < 0.30}
fmt_txt = ", ".join("%s %s" % (c, num(v[0]))
                    for c, v in sorted(sostienen.items(), key=lambda p: -p[1][0]))
fmt_no_txt = ("; no se sostiene en " +
              ", ".join("%s (%s, con sólo %d folios)" % (c, num(v[0]), v[1])
                        for c, v in no_sost.items())) if no_sost else ""

filas = sorted(((v, FAMILIA_STYLE[v], r_s["por_clase"][v], r_s["n_por_clase"][v])
                for v in r_s["n_por_clase"] if r_s["n_por_clase"][v] >= 14),
               key=lambda f: -f[2])
tabla = "\n".join("| `%s` | %s | %s | %d |" % (v, ("**%s**" % f if f != "movimiento" else f), num(p), n)
                  for v, f, p, n in filas)

display(Markdown(f"""
### En dos oraciones — que es lo único que el oral necesita

> **El espacio CLIP separa el tema con claridad y no separa el estilo:** el κ del vecino es
> **{num(r_t['kappa'])}** para el tema derivado del título —retratos {num(r_t['por_clase']['retrato'])},
> paisajes {num(r_t['por_clase']['paisaje'])}— contra **{num(r_s['kappa'])}** para los
> {r_s['n_clases']} valores de `style_title`, y sobre **las mismas {h2h_t['n']} obras** que tienen
> las dos etiquetas la diferencia es **{num(h2h_t['kappa'])} contra {num(h2h_s['kappa'])}**.
>
> **Y el tema no es la cultura disfrazada:** sigue separando *dentro* de cada tradición
> ({cult_txt}), y los {int((tema == 'paisaje').sum())} paisajes que se juntan son
> {pais_txt} a la vez.

**el objetivo opcional de la figura UMAP se cumple en la dirección de la hipótesis**, y con eso la figura es lo que
`la teoría aplicada § UMAP` prometía: *la evidencia visual del desafío central del TP*. Un buscador visual
puro sobre este espacio recupera **el mismo tema** — que es exactamente por qué no sirve para
*"versiones más cálidas"*, y por qué existen el pre-retrieval y la rama de texto.

### Lo que el estilo sí separa, y por qué no salva al estilo

`style` da κ = {num(r_s['kappa'])} con los {r_s['n_clases']} valores y **{num(r_mov['kappa'])} si se
lo restringe a los {r_mov['n_clases']} que son movimientos de verdad**. Mirado por clase, la señal
está concentrada en lugares que **no** sostienen la hipótesis contraria:

| Valor de `style_title` | Familia | Pureza k-NN | n |
|---|---|---:|---:|
{tabla}

**Los valores de `style` que se separan son, casi todos, culturas con el nombre cambiado**, y el
único movimiento que se separa de verdad es el Impresionismo — el más homogéneo visualmente. Los
**períodos no se separan en absoluto**. Es un resultado más fuerte que *"el estilo no separa"*,
porque dice **por qué**: `style_title` sólo discrimina donde encubre otra variable.

### 🔴 Dos hallazgos que nadie pidió

**1 · El etiquetado con más separación después del tema es el FORMATO, no el estilo ni la
cultura.** Folio de álbum o manuscrito vs. pintura de caballete da **κ = {num(r_f['kappa'])}** —
por encima de `culture` ({num(r_c['kappa'])}) y muy por encima de `style` ({num(r_s['kappa'])})— y
**se sostiene dentro de cada cultura** donde hay masa de los dos ({fmt_txt}{fmt_no_txt}), así que no es la
cultura disfrazada. Y tiene consecuencia operativa: en la rama visual el formato funciona como
**un filtro previo que nadie declaró**. Quien suba una pintura de caballete va a recibir pinturas
de caballete y casi ningún folio, para cualquier pedido. Puede ser deseable; hoy no es una
decisión, es un efecto.

**2 · El eje macro de la proyección no es ni tema ni estilo: es la tradición cultural.** Los dos
bloques grandes de la nube son Asia (China, Japón, India mogol) y Europa/América (Francia, Italia,
Países Bajos, EE. UU.), con `culture` en κ = {num(r_c['kappa'])} y las purezas más altas en
Mughal India ({num(r_c['por_clase']['Mughal India'])}), China ({num(r_c['por_clase']['China'])}) y
Japón ({num(r_c['por_clase']['Japan'])}). Es **buena noticia para el producto**: es la evidencia de
que un pedido como *"el mismo tema pero en la cultura japonesa"* —el ejemplo literal del email
aprobado— tiene asidero geométrico en el espacio donde se busca.

**Y un control que da el resultado tranquilizador:** `museo` es el etiquetado **menos** separado
de los seis (κ = {num(r_m['kappa'])}). Los dos museos no forman clusters propios, así que la
estructura que se ve **no es un artefacto de haber mezclado dos colecciones**.
"""))


### En dos oraciones — que es lo único que el oral necesita

> **El espacio CLIP separa el tema con claridad y no separa el estilo:** el κ del vecino es
> **0,67** para el tema derivado del título —retratos 0,79,
> paisajes 0,77— contra **0,35** para los
> 39 valores de `style_title`, y sobre **las mismas 189 obras** que tienen
> las dos etiquetas la diferencia es **0,68 contra 0,22**.
>
> **Y el tema no es la cultura disfrazada:** sigue separando *dentro* de cada tradición
> (Italy 0,70 · Netherlands 0,66 · China 0,60 · France 0,60 · Japan 0,50 · Mughal India 0,32), y los 172 paisajes que se juntan son
> China (62), Japan (43), France (13), Netherlands (10), Mughal India (6), United States (4), None (3), Italy (2), Northern India (1) a la vez.

**el objetivo de separación por tema se cumple en la dirección de la hipótesis**, y con eso la figura es lo que
la figura prometía: *la evidencia visual del desafío central del TP*. Un buscador visual
puro sobre este espacio recupera **el mismo tema** — que es exactamente por qué no sirve para
*"versiones más cálidas"*, y por qué existen el pre-retrieval y la rama de texto.

### Lo que el estilo sí separa, y por qué no salva al estilo

`style` da κ = 0,35 con los 39 valores y **0,42 si se
lo restringe a los 13 que son movimientos de verdad**. Mirado por clase, la señal
está concentrada en lugares que **no** sostienen la hipótesis contraria:

| Valor de `style_title` | Familia | Pureza k-NN | n |
|---|---|---:|---:|
| `Himalayan` | **cultura** | 0,82 | 23 |
| `Chinese (culture or style)` | **cultura** | 0,75 | 28 |
| `Japanese (culture or style)` | **cultura** | 0,70 | 29 |
| `Impressionism` | movimiento | 0,70 | 111 |
| `15th century` | **período** | 0,41 | 25 |
| `17th Century` | **período** | 0,39 | 80 |
| `Post-Impressionism` | movimiento | 0,28 | 32 |
| `16th Century` | **período** | 0,26 | 33 |
| `Renaissance` | movimiento | 0,24 | 21 |
| `Realism` | movimiento | 0,20 | 26 |
| `18th Century` | **período** | 0,20 | 38 |
| `nineteenth century` | **período** | 0,12 | 28 |
| `19th century` | **período** | 0,06 | 25 |
| `sixteenth century` | **período** | 0,06 | 14 |

**Los valores de `style` que se separan son, casi todos, culturas con el nombre cambiado**, y el
único movimiento que se separa de verdad es el Impresionismo — el más homogéneo visualmente. Los
**períodos no se separan en absoluto**. Es un resultado más fuerte que *"el estilo no separa"*,
porque dice **por qué**: `style_title` sólo discrimina donde encubre otra variable.

### 🔴 Dos hallazgos que nadie pidió

**1 · El etiquetado con más separación después del tema es el FORMATO, no el estilo ni la
cultura.** Folio de álbum o manuscrito vs. pintura de caballete da **κ = 0,66** —
por encima de `culture` (0,46) y muy por encima de `style` (0,35)— y
**se sostiene dentro de cada cultura** donde hay masa de los dos (China 0,54, otras 0,52, Mughal India 0,49; no se sostiene en Northern India (0,02, con sólo 20 folios)), así que no es la
cultura disfrazada. Y tiene consecuencia operativa: en la rama visual el formato funciona como
**un filtro previo que nadie declaró**. Quien suba una pintura de caballete va a recibir pinturas
de caballete y casi ningún folio, para cualquier pedido. Puede ser deseable; hoy no es una
decisión, es un efecto.

**2 · El eje macro de la proyección no es ni tema ni estilo: es la tradición cultural.** Los dos
bloques grandes de la nube son Asia (China, Japón, India mogol) y Europa/América (Francia, Italia,
Países Bajos, EE. UU.), con `culture` en κ = 0,46 y las purezas más altas en
Mughal India (0,80), China (0,73) y
Japón (0,67). Es **buena noticia para el producto**: es la evidencia de
que un pedido como *"el mismo tema pero en la cultura japonesa"* —el ejemplo literal del email
aprobado— tiene asidero geométrico en el espacio donde se busca.

**Y un control que da el resultado tranquilizador:** `museo` es el etiquetado **menos** separado
de los seis (κ = 0,30). Los dos museos no forman clusters propios, así que la
estructura que se ve **no es un artefacto de haber mezclado dos colecciones**.


### Las limitaciones, dichas

- **La etiqueta de tema es cruda y cubre una fracción del corpus.** Es el precio de que sea
  independiente de CLIP, y es el que `el registro de decisiones el criterio de idioma` pide pagar: *"una etiqueta independiente sobre
  el 25 % de las obras prueba más que una etiqueta circular sobre el 100 %"*.
- **Quedan falsos positivos, y no se corrigieron a mano a propósito.** El que vimos al auditar
  la muestra es `"The Eighteen Sages of the Lotus Society"` en `flora y fauna` (el *"Lotus"* es
  una cofradía). La familia de los topónimos franceses —`Gare Saint-Lazare`— sí se arregló,
  pero **con una regla** (`\bsaints?\b(?!-)`), no con una lista de excepciones: una lista curada
  obra por obra dejaría de ser auditable y volvería la etiqueta tan discrecional como la
  circular que la los tres ciclos de vida descarta.
- **`desnudo` tiene n = 8.** Se muestra con su `n` porque esconder una clase chica es peor, pero
  su pureza no significa nada con ese tamaño. Lo mismo vale para los controles marcados ⚠️.
- **El cabeza a cabeza (el contrato de salida) son menos de 200 obras, todas de AIC** — el `n` exacto lo emite la
  celda, no este párrafo. Es la comparación limpia que hay —`style` no existe en CMA—, no una
  muestra grande; a cambio controla el museo por construcción.
- **κ del vecino no es una métrica estándar de la literatura.** Es una lectura simple del
  espacio, construida acá, y hay que presentarla así — igual que `los criterios de entrega los contratos` pide decir que
  los tamaños de muestra del proyecto son presupuesto de tiempo humano y no potencia
  estadística. Lo que la sostiene es que **se mide en 512 dimensiones y no en la proyección**, y
  que el κ en 2-d la acompaña (la decisión de separar la evaluación).
- **No hay variante registrada**, así que el panel de `culture` usa el criterio de
  `config/default.toml` y no el de una receta. Ver el recuadro del encabezado.